In [ ]:
# NOTE:
# 1) The input data created through this script does not represent the training data of the model exactly, since the model has columns with delta values for each metric.
# 2) So delta value will not be included in the input data, since the repository csv files from arcan have only 1 version of the repository
# 3) Even so, the script considers if there are multiple versions involved in the csv files, then the delta values can be created
# 4) The model can only give approximate suggestions, since the input data is not completely mirrored in the training data
# 5) The script makes sure the only cyclic dependencies, hub like dependencies and unstable dependencies are parsed from the csv files
# 6) The repository used is logging-log4j2 detected smells using Arcan

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

COMPONENT_METRICS_PATH = Path(r"baseline_csv/component-metrics.csv")
SMELL_CHARACTERISTICS_PATH = Path(r"baseline_csv/smell-characteristics.csv")
SMELL_AFFECTS_PATH = Path(r"baseline_csv/smell-affects.csv")

OUTPUT_INPUTS_PATH = Path(r"C:\dsarp_outputs\logging_log4j2_model_inputs_from_csv.csv")
OUTPUT_INPUTS_PATH.parent.mkdir(parents=True, exist_ok=True)

SELECTED_SMELLS = {
    "cyclicDep": "Cyclic Dependency",
    "hubLikeDep": "Hub-like Dependency",
    "unstableDep": "Unstable Dependency",
}

PRIMARY_METRIC_BY_SMELL = {
    "Cyclic Dependency": "cyclic_packages",
    "Hub-like Dependency": "hub_like_packages",
    "Unstable Dependency": "unstable_dependencies",
}

METRIC_FIELDS = [
    "java_files",
    "packages",
    "package_edges",
    "cyclic_packages",
    "max_fan_in",
    "max_fan_out",
    "unstable_dependencies",
    "hub_like_packages",
    "large_packages",
    "median_package_size",
]

In [ ]:
# Load CSV files

component_metrics = pd.read_csv(COMPONENT_METRICS_PATH)
smell_characteristics = pd.read_csv(SMELL_CHARACTERISTICS_PATH)
smell_affects = pd.read_csv(SMELL_AFFECTS_PATH)

print(component_metrics.shape)
print(smell_characteristics.shape)
print(smell_affects.shape)

display(component_metrics.head())
display(smell_characteristics.head())
display(smell_affects.head())

In [ ]:
# Normalize smell-affects columns.
# Some exports have blank first column names, so we standardize them.

def normalize_smell_affects_columns(df):
    df = df.copy()

    # If the first two headers are unnamed, this matches the format you showed earlier.
    if str(df.columns[0]).startswith("Unnamed") or df.columns[0] == "":
        df.columns = [
            "project",
            "versionId",
            "versionIndex",
            "versionDate",
            "edgeId",
            "relation",
            "smellType",
            "smellVertexId",
            "affectedElement",
            "affectedElementId",
        ]
    else:
        # If already named differently, keep as much as possible.
        rename_map = {
            "from": "smellType",
            "fromId": "smellVertexId",
            "to": "affectedElement",
            "toId": "affectedElementId",
        }
        df = df.rename(columns=rename_map)

    return df


smell_affects = normalize_smell_affects_columns(smell_affects)

display(smell_affects.head())

In [ ]:
# Keep only selected architecture smells

smells = smell_characteristics.copy()
smells = smells[smells["smellType"].isin(SELECTED_SMELLS.keys())].copy()

smells["architecture_smell"] = smells["smellType"].map(SELECTED_SMELLS)

print("Selected smell rows:", len(smells))
display(smells["architecture_smell"].value_counts())

In [ ]:
# Aggregate affected elements per smell instance

affects_selected = smell_affects[
    smell_affects["smellType"].isin(SELECTED_SMELLS.keys())
].copy()

affects_selected["smellVertexId"] = affects_selected["smellVertexId"].astype(str)
smells["vertexId"] = smells["vertexId"].astype(str)

affected_by_smell = (
    affects_selected
    .groupby(["project", "versionId", "smellType", "smellVertexId"])["affectedElement"]
    .apply(lambda values: "|".join(sorted(set(str(v) for v in values if pd.notna(v)))))
    .reset_index()
)

affected_by_smell = affected_by_smell.rename(columns={
    "smellVertexId": "vertexId",
    "affectedElement": "affected_elements",
})

smells = smells.merge(
    affected_by_smell,
    on=["project", "versionId", "smellType", "vertexId"],
    how="left",
)

smells["affected_elements"] = smells["affected_elements"].fillna(
    smells.get("AffectedElements", "").astype(str)
)

display(smells[["project", "versionId", "architecture_smell", "affected_elements"]].head())

In [ ]:
# Compute version-level metrics from component-metrics.csv.
# These are shaped to resemble the metrics used during training.

def compute_version_metrics(component_df, smell_df):
    rows = []

    grouped = component_df.groupby(["project", "versionId"])

    for (project, version_id), group in grouped:
        package_rows = group[
            (group.get("ComponentType", "") == "PACKAGE") |
            (group.get("constructType", "") == "PACKAGE")
        ].copy()

        if package_rows.empty:
            package_rows = group.copy()

        fan_in = pd.to_numeric(package_rows.get("FanIn", 0), errors="coerce").fillna(0)
        fan_out = pd.to_numeric(package_rows.get("FanOut", 0), errors="coerce").fillna(0)
        instability = pd.to_numeric(package_rows.get("InstabilityMetric", 0), errors="coerce").fillna(0)
        loc = pd.to_numeric(package_rows.get("LinesOfCode", 0), errors="coerce").fillna(0)

        packages = len(package_rows)
        max_fan_in = int(fan_in.max()) if len(fan_in) else 0
        max_fan_out = int(fan_out.max()) if len(fan_out) else 0

        hub_like_packages = int(((fan_in >= 8) & (fan_out >= 8)).sum())
        unstable_dependencies = int(((fan_in >= 5) & (instability > 0.8)).sum())

        median_package_size = float(loc.median()) if len(loc) else 0.0
        large_threshold = max(10.0, median_package_size * 2.0)
        large_packages = int((loc >= large_threshold).sum())

        smell_subset = smell_df[
            (smell_df["project"] == project) &
            (smell_df["versionId"] == version_id)
        ]

        cyclic_packages = int((smell_subset["smellType"] == "cyclicDep").sum())

        # package_edges is approximated from detector NumberOfEdges where available.
        if "NumberOfEdges" in smell_subset.columns:
            package_edges = pd.to_numeric(
                smell_subset["NumberOfEdges"],
                errors="coerce",
            ).fillna(0).sum()
        else:
            package_edges = 0

        rows.append({
            "project": project,
            "versionId": version_id,
            "java_files": 0,  # not available from these CSVs
            "packages": packages,
            "package_edges": float(package_edges),
            "cyclic_packages": cyclic_packages,
            "max_fan_in": max_fan_in,
            "max_fan_out": max_fan_out,
            "unstable_dependencies": unstable_dependencies,
            "hub_like_packages": hub_like_packages,
            "large_packages": large_packages,
            "median_package_size": median_package_size,
        })

    return pd.DataFrame(rows)


version_metrics = compute_version_metrics(component_metrics, smell_characteristics)

display(version_metrics.head())

In [ ]:
# If multiple versions exist, compute deltas against previous version.
# If only one version exists, use before=current, after=current, delta=0.
# This keeps the input format close to what the model saw during training.

def add_before_after_deltas(version_metrics):
    version_metrics = version_metrics.copy()

    if "versionIndex" in component_metrics.columns:
        version_order = (
            component_metrics[["project", "versionId", "versionIndex"]]
            .drop_duplicates()
        )
        version_metrics = version_metrics.merge(
            version_order,
            on=["project", "versionId"],
            how="left",
        )
    else:
        version_metrics["versionIndex"] = 1

    version_metrics = version_metrics.sort_values(["project", "versionIndex"])

    output_rows = []

    for project, group in version_metrics.groupby("project"):
        group = group.sort_values("versionIndex").reset_index(drop=True)

        for idx, row in group.iterrows():
            previous = group.iloc[idx - 1] if idx > 0 else row

            out = {
                "project": row["project"],
                "versionId": row["versionId"],
                "versionIndex": row["versionIndex"],
            }

            for field in METRIC_FIELDS:
                before_value = previous[field]
                after_value = row[field]
                delta = before_value - after_value

                out[f"{field}_before"] = before_value
                out[f"{field}_after"] = after_value
                out[f"{field}_delta"] = delta

            output_rows.append(out)

    return pd.DataFrame(output_rows)


version_deltas = add_before_after_deltas(version_metrics)

display(version_deltas.head())

In [ ]:
# Merge smell rows with version-level metric deltas

model_inputs = smells.merge(
    version_deltas,
    on=["project", "versionId"],
    how="left",
)

model_inputs["primary_metric"] = model_inputs["architecture_smell"].map(PRIMARY_METRIC_BY_SMELL)

model_inputs["primary_metric_before"] = model_inputs.apply(
    lambda row: row.get(f"{row['primary_metric']}_before", 0),
    axis=1,
)

model_inputs["primary_metric_after"] = model_inputs.apply(
    lambda row: row.get(f"{row['primary_metric']}_after", 0),
    axis=1,
)

model_inputs["primary_metric_delta"] = model_inputs.apply(
    lambda row: row.get(f"{row['primary_metric']}_delta", 0),
    axis=1,
)

display(model_inputs[[
    "project",
    "versionId",
    "architecture_smell",
    "primary_metric",
    "primary_metric_before",
    "primary_metric_after",
    "primary_metric_delta",
    "affected_elements",
]].head())

In [ ]:
# we can confirm since the primary_metric_Delta is 0, only a single version was considered

In [ ]:
# Add improved/worsened metric summaries, matching the training input style

def metric_summary_for_row(row):
    improved = []
    worsened = []

    for field in METRIC_FIELDS:
        delta = row.get(f"{field}_delta", 0)

        try:
            delta = float(delta)
        except Exception:
            delta = 0

        if delta > 0:
            improved.append(field)
        elif delta < 0:
            worsened.append(field)

    return pd.Series({
        "improved_metric_names": "|".join(improved),
        "worsened_metric_names": "|".join(worsened),
        "improved_metric_count": len(improved),
        "worsened_metric_count": len(worsened),
    })


summary_cols = model_inputs.apply(metric_summary_for_row, axis=1)
model_inputs = pd.concat([model_inputs, summary_cols], axis=1)

display(model_inputs[[
    "architecture_smell",
    "improved_metric_names",
    "worsened_metric_names",
    "improved_metric_count",
    "worsened_metric_count",
]].head())

In [ ]:
# the columns of improved metric count etc would also be empty, since there is only a single version of the logging-log4j2 software system

In [ ]:
# Build input_text compatible with the improved DistilBERT training format

def build_model_input_text(row):
    metric_parts = []

    for field in METRIC_FIELDS:
        before_value = row.get(f"{field}_before", 0)
        after_value = row.get(f"{field}_after", 0)
        delta = row.get(f"{field}_delta", 0)

        metric_parts.append(
            f"{field} changed from {before_value} to {after_value}, delta {delta}."
        )

    affected = str(row.get("affected_elements", "")).replace("|", ", ")

    # Extra smell-detector fields, if available
    detector_parts = []

    for col in [
        "Severity",
        "Size",
        "Shape",
        "NumberOfEdges",
        "SmellExtent",
        "Strength",
        "ATDI",
        "ATDI_WEIGHTED",
        "InstabilityGap",
        "LOCDensity",
    ]:
        if col in row.index and pd.notna(row[col]):
            detector_parts.append(f"{col}: {row[col]}.")

    return (
        f"Architecture smell: {row['architecture_smell']}. "
        f"Primary metric: {row['primary_metric']}. "
        f"Primary metric changed from {row['primary_metric_before']} "
        f"to {row['primary_metric_after']}, "
        f"delta {row['primary_metric_delta']}. "
        f"Affected elements: {affected}. "
        f"Improved metrics: {row['improved_metric_names']}. "
        f"Worsened metrics: {row['worsened_metric_names']}. "
        f"Improved metric count: {row['improved_metric_count']}. "
        f"Worsened metric count: {row['worsened_metric_count']}. "
        + " ".join(detector_parts)
        + " "
        + " ".join(metric_parts)
    )


model_inputs["input_text"] = model_inputs.apply(build_model_input_text, axis=1)

display(model_inputs[[
    "project",
    "versionId",
    "architecture_smell",
    "affected_elements",
    "input_text",
]].head())

display(model_inputs.head())

In [ ]:
for col in model_inputs.columns:
    print(col)

In [ ]:
# Save parsed model inputs

model_inputs.to_csv(OUTPUT_INPUTS_PATH, index=False)

model_inputs.to_json(
    OUTPUT_INPUTS_PATH.with_suffix(".jsonl"),
    orient="records",
    lines=True,
)

print("Saved model inputs to:", OUTPUT_INPUTS_PATH)
print("Rows:", len(model_inputs))